# Mid Project Dashboard: Financial Literacy and Retirement Readiness

This notebook is the build pipeline. Running it generates **index.html**, a standalone
single-page dashboard, and opens it in your browser.

**How to run:** Run > Run All Cells. The last cell writes `index.html` next to this notebook
and opens it. On GitHub Pages the same file becomes the live site.

**Structure of the page:**
1. Financial Literacy Quiz (10 questions). No visualization is revealed until the user
   answers all questions and submits.
2. After submission the dashboard replaces the quiz: Overview, Demographics, Financial
   Behavior, Retirement Readiness, Your Profile, and a placeholder Model tab.

**Data contract:** the visualizations use the raw survey data as-is. No missing values are
removed, no rows are dropped, no reserved codes are replaced. Reserved codes 888888 (don't
know / refused) and 999999 (not applicable / filter) appear in the charts as their own
categories where they occur.

In [ ]:
# 1. Environment check and imports
import sys
import importlib

print(f"Python {sys.version.split()[0]}")
REQUIRED = ["numpy", "pandas", "plotly", "openpyxl", "folium", "wordcloud",
            "PIL", "matplotlib"]
OPTIONAL = ["pyarrow"]  # pyarrow: fast data cache
missing = []
for pkg in REQUIRED + OPTIONAL:
    try:
        mod = importlib.import_module(pkg)
        tag = " (optional)" if pkg in OPTIONAL else ""
        print(f"  {pkg:<10} {getattr(mod, '__version__', 'installed')}{tag}")
    except ImportError:
        if pkg in REQUIRED:
            missing.append(pkg)
            print(f"  {pkg:<10} MISSING (required)")
        else:
            print(f"  {pkg:<10} not installed (optional)")
if missing:
    raise ImportError("Missing required packages. Run: pip install " + " ".join(missing))

import os
import io
import json
import base64
import webbrowser
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs
import folium
from wordcloud import WordCloud

px.defaults.template = "plotly_white"

In [ ]:
# 2. Load data and define code -> label maps (used only for axis labels; raw values kept)

DATA_PATH = r"data/Social_Survey/data_24.xlsx"
CACHE_PATH = r"data/Social_Survey/data_24_cache.parquet"

if os.path.exists(CACHE_PATH):
    df = pd.read_parquet(CACHE_PATH)
else:
    df = pd.read_excel(DATA_PATH)
    try:
        df.to_parquet(CACHE_PATH)
    except Exception:
        pass
print(f"Social Survey 2024 loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

AGE_LABELS = {1: "20-24", 2: "25-29", 3: "30-34", 4: "35-39", 5: "40-44", 6: "45-49",
              7: "50-54", 8: "55-59", 9: "60-64", 10: "65-74", 11: "75+"}
INCOME_LABELS = {1: "up to 2,000", 2: "2,001-4,000", 3: "4,001-6,000", 4: "6,001-8,000",
                 5: "8,001-10,000", 6: "10,001-14,000", 7: "14,001-19,000",
                 8: "19,001-27,000", 9: "27,001-40,000", 10: "40,001+"}
SAT_LABELS = {1: "Very satisfied", 2: "Satisfied", 3: "Not so satisfied", 4: "Not satisfied at all"}
LEFTOVER_LABELS = {1: "A lot is left over", 2: "Some is left over", 3: "Roughly breaks even",
                   4: "Runs short some months", 5: "Runs short every month"}
OPTIMISM_LABELS = {1: "Better off", 2: "About the same", 3: "Worse off"}

# CBS Nafa (subdistrict) codes + approximate center coordinates for the Israel map
NAFA = {
    11: {"name": "Jerusalem",           "lat": 31.7683, "lon": 35.2137},
    21: {"name": "Safed (Tzfat)",       "lat": 32.9646, "lon": 35.4960},
    22: {"name": "Kinneret",            "lat": 32.7196, "lon": 35.5497},
    23: {"name": "Yizrael",             "lat": 32.6111, "lon": 35.2892},
    24: {"name": "Akko",                "lat": 32.9281, "lon": 35.0817},
    29: {"name": "Golan",               "lat": 32.9915, "lon": 35.6952},
    31: {"name": "Haifa",               "lat": 32.7940, "lon": 34.9896},
    32: {"name": "Hadera",              "lat": 32.4380, "lon": 34.9195},
    41: {"name": "Sharon",              "lat": 32.3216, "lon": 34.8532},
    42: {"name": "Petah Tikva",         "lat": 32.0870, "lon": 34.8878},
    43: {"name": "Ramla",               "lat": 31.9293, "lon": 34.8664},
    44: {"name": "Rehovot",             "lat": 31.8942, "lon": 34.8103},
    51: {"name": "Tel Aviv",            "lat": 32.0853, "lon": 34.7818},
    61: {"name": "Ashkelon",            "lat": 31.6688, "lon": 34.5748},
    62: {"name": "Beer Sheva",          "lat": 31.2530, "lon": 34.7915},
    71: {"name": "Judea and Samaria",   "lat": 32.0857, "lon": 35.1000},
}

BEHAVIOR_ITEMS = [
    ("HisachonPensyoni", "Has a pension savings account"),
    ("HatavaKHishtalmut_wp", "Has Study Fund through employer"),
    ("HashkaotBank", "Invests through a bank"),
    ("HalvaaBank", "Has a bank loan"),
    ("InternetDohPensia", "Checks pension statement online"),
    ("InternetDohAshrai", "Checks credit report online"),
    ("InternetShilemMatbeaDig", "Paid with digital currency (crypto)"),
]

In [ ]:
# 3. Quiz definition. Population Yes-share attached to each behavior question is the
# share among ALL 6,907 respondents (raw), not among a filtered subset.

QUIZ = [
    {"id": "Q1", "kind": "knowledge",
     "text": "What is a typical annual management fee (dmei nihul) on a Study Fund (Keren Hishtalmut) in Israel?",
     "options": ["0.05% to 0.3%", "0.5% to 1%", "2% to 3%", "5% to 10%"], "correct": 1},
    {"id": "Q2", "kind": "knowledge",
     "text": "After how many years can a Study Fund be withdrawn tax free in the standard case (not retirement)?",
     "options": ["3 years", "5 years", "6 years", "10 years"], "correct": 2},
    {"id": "Q3", "kind": "behavior", "survey_var": "HisachonPensyoni", "yes_code": 1,
     "text": "Do you currently have a pension savings account?", "options": ["Yes", "No"]},
    {"id": "Q4", "kind": "behavior", "survey_var": "HatavaKHishtalmut_wp", "yes_code": 1,
     "text": "Do you have a Study Fund (Keren Hishtalmut) through your employer?", "options": ["Yes", "No"]},
    {"id": "Q5", "kind": "knowledge",
     "text": "What is Bitcoin?",
     "options": ["An Israeli technology stock", "A digital, decentralized currency",
                 "A government-issued bond", "A commodity ETF"], "correct": 1},
    {"id": "Q6", "kind": "behavior", "survey_var": "InternetShilemMatbeaDig", "yes_code": 1,
     "text": "Have you ever paid for something using a digital currency (for example Bitcoin)?",
     "options": ["Yes", "No"]},
    {"id": "Q7", "kind": "behavior", "survey_var": "HashkaotBank", "yes_code": 1,
     "text": "Do you invest through a bank (stocks, bonds, mutual funds)?", "options": ["Yes", "No"]},
    {"id": "Q8", "kind": "knowledge",
     "text": "If annual inflation is 5% and your savings account pays 3% interest, your real return is:",
     "options": ["+8%", "+2%", "minus 2%", "0%"], "correct": 2},
    {"id": "Q9", "kind": "behavior", "survey_var": "InternetDohPensia", "yes_code": 1,
     "text": "Do you check your pension statement online at least once a year?", "options": ["Yes", "No"]},
    {"id": "Q10", "kind": "knowledge",
     "text": "What best describes compound interest?",
     "options": ["Interest deducted from the principal each year",
                 "Interest earned also earns interest over time",
                 "A fixed dollar amount added yearly",
                 "A one time bonus paid at maturity"], "correct": 1},
]
for q in QUIZ:
    if q["kind"] == "behavior":
        # Raw share of Yes among ALL respondents (denominator = total N).
        q["popYes"] = round(float((df[q["survey_var"]] == q["yes_code"]).mean() * 100), 1)

print("Quiz ready:", len(QUIZ), "questions")

In [ ]:
# 4. Build all Plotly figures for the dashboard. Data used exactly as-is.

FIGS = {}

def _clean(fig, height=430):
    """Remove all gridlines and zero-lines; set consistent typography and margins."""
    fig.update_layout(
        height=height, margin=dict(t=70, r=30, b=70, l=60),
        font=dict(family="Segoe UI, Arial", size=13),
        title=dict(font=dict(size=17), x=0.5, xanchor="center"),
        plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.update_xaxes(showgrid=False, zeroline=False, showline=True,
                     linecolor="#d1d5db", ticks="outside", tickcolor="#d1d5db")
    fig.update_yaxes(showgrid=False, zeroline=False, showline=True,
                     linecolor="#d1d5db", ticks="outside", tickcolor="#d1d5db")
    return fig

def _label_v(fig, series):
    """Attach value labels above vertical bars."""
    fig.update_traces(text=[f"{int(v):,}" for v in series], textposition="outside",
                      cliponaxis=False, textfont=dict(size=12))
    return fig

# --- Demographics
gender_counts = (df["Minn"].map({1: "Male", 2: "Female"}).fillna("Other/unspecified")
                 .value_counts().rename_axis("Gender").reset_index(name="Count"))
fig = px.pie(gender_counts, names="Gender", values="Count",
             title=f"Gender distribution (raw, n = {len(df):,})",
             color_discrete_sequence=["#2c7be5", "#f6b73c", "#9CA3AF"], hole=0.45)
fig.update_traces(textinfo="percent+label+value", textfont_size=13)
FIGS["fig-gender"] = _clean(fig)

age_order = [AGE_LABELS[k] for k in sorted(AGE_LABELS)]
age_counts = (df["Gil"].map(AGE_LABELS).value_counts().reindex(age_order)
              .rename_axis("Age group").reset_index(name="Respondents"))
fig = px.bar(age_counts, x="Age group", y="Respondents",
             title="Age distribution (all respondents)",
             color="Respondents", color_continuous_scale="Blues")
fig.update_layout(coloraxis_showscale=False)
_label_v(fig, age_counts["Respondents"])
FIGS["fig-age"] = _clean(fig)

inc_order_lbl = [INCOME_LABELS[k] for k in sorted(INCOME_LABELS)] + ["Don't know (888888)", "Not applicable (999999)"]
inc_series_raw = df["HachnasaKoleletNeto"].map(
    {**INCOME_LABELS, 888888: "Don't know (888888)", 999999: "Not applicable (999999)"})
inc_counts = (inc_series_raw.value_counts().reindex(inc_order_lbl)
              .rename_axis("Net household income (NIS)").reset_index(name="Respondents"))
fig = px.bar(inc_counts, x="Net household income (NIS)", y="Respondents",
             title="Household net income distribution (raw, including reserved codes)",
             color="Respondents", color_continuous_scale="Greens")
fig.update_layout(coloraxis_showscale=False, xaxis_tickangle=-30)
_label_v(fig, inc_counts["Respondents"])
FIGS["fig-income"] = _clean(fig, height=490)

edu = pd.DataFrame({
    "Education milestone": ["Bachelor (BA)", "Master (MA)", "Doctorate (PhD)",
                             "Post secondary", "Matriculation certificate"],
    "Respondents": [int((df["Ba"] == 1).sum()), int((df["Ma"] == 1).sum()),
                    int((df["Phd"] == 1).sum()), int((df["Post_Secondary"] == 1).sum()),
                    int((df["Matriculation_Certificate"] == 1).sum())],
}).sort_values("Respondents")
fig = px.bar(edu, x="Respondents", y="Education milestone", orientation="h",
             title="Education milestones held (not mutually exclusive)",
             color="Respondents", color_continuous_scale="Purples")
fig.update_layout(coloraxis_showscale=False)
fig.update_traces(text=[f"{int(v):,}" for v in edu["Respondents"]],
                  textposition="outside", cliponaxis=False, textfont=dict(size=12))
FIGS["fig-edu"] = _clean(fig)

# Age x Income HEATMAP (raw counts, all codes preserved as columns/rows)
ages_labeled = df["Gil"].map({**AGE_LABELS, 888888: "DK", 999999: "NA"})
incs_labeled = df["HachnasaKoleletNeto"].map({**INCOME_LABELS, 888888: "DK", 999999: "NA"})
crosstab = pd.crosstab(ages_labeled, incs_labeled)
row_order = age_order + [x for x in ["DK", "NA"] if x in crosstab.index]
col_order = [INCOME_LABELS[k] for k in sorted(INCOME_LABELS)] + [x for x in ["DK", "NA"] if x in crosstab.columns]
crosstab = crosstab.reindex(index=[r for r in row_order if r in crosstab.index],
                             columns=[c for c in col_order if c in crosstab.columns])
fig = px.imshow(crosstab, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                labels=dict(x="Net household income (NIS)", y="Age group", color="Count"),
                title="Age x Income heatmap (respondent counts, raw)")
fig.update_layout(xaxis_tickangle=-30)
FIGS["fig-heatmap"] = _clean(fig, height=520)

# --- Financial Behavior
records = []
for var, label in BEHAVIOR_ITEMS:
    yes_pct = round(float((df[var] == 1).mean() * 100), 1)
    records.append({"Behavior": label, "Yes %": yes_pct})
behavior_df = pd.DataFrame(records).sort_values("Yes %")
fig = px.bar(behavior_df, x="Yes %", y="Behavior", orientation="h",
             title="Financial behaviors (share answering Yes, raw denominator = all respondents)",
             color="Yes %", color_continuous_scale="Teal")
fig.update_layout(coloraxis_showscale=False, xaxis_ticksuffix="%")
fig.update_traces(text=[f"{v}%" for v in behavior_df["Yes %"]],
                  textposition="outside", cliponaxis=False, textfont=dict(size=12))
FIGS["fig-behaviors"] = _clean(fig, height=460)

sat_map_raw = {**SAT_LABELS, 888888: "Don't know (888888)"}
sat_order = [SAT_LABELS[k] for k in sorted(SAT_LABELS)] + ["Don't know (888888)"]
sat_counts = (df["MerutzeTichnunFinPrisha"].map(sat_map_raw)
              .value_counts().reindex(sat_order)
              .rename_axis("Satisfaction").reset_index(name="Respondents"))
fig = px.bar(sat_counts, x="Satisfaction", y="Respondents",
             title="Satisfaction with retirement financial planning (raw)",
             color="Respondents", color_continuous_scale="RdYlGn_r")
fig.update_layout(coloraxis_showscale=False)
_label_v(fig, sat_counts["Respondents"])
FIGS["fig-satisfaction"] = _clean(fig)

lo_map_raw = {**LEFTOVER_LABELS, 888888: "Don't know (888888)"}
lo_order = [LEFTOVER_LABELS[k] for k in sorted(LEFTOVER_LABELS)] + ["Don't know (888888)"]
lo_counts = (df["NisharKesef"].map(lo_map_raw).value_counts().reindex(lo_order)
             .rename_axis("End of month status").reset_index(name="Respondents"))
fig = px.bar(lo_counts, x="End of month status", y="Respondents",
             title="Financial slack at end of month (raw)",
             color="Respondents", color_continuous_scale="RdYlGn_r")
fig.update_layout(coloraxis_showscale=False, xaxis_tickangle=-15)
_label_v(fig, lo_counts["Respondents"])
FIGS["fig-leftover"] = _clean(fig)

# --- Retirement Readiness
by_age_series = df.groupby(df["Gil"].map(AGE_LABELS))["MerutzeTichnunFinPrisha"].apply(
    lambda s: (s.isin([3, 4]).sum() / len(s) * 100)).reindex(age_order)
by_age_df = by_age_series.rename_axis("Age group").reset_index(name="Not satisfied (%)")
fig = px.line(by_age_df, x="Age group", y="Not satisfied (%)", markers=True,
              text=[f"{v:.0f}%" for v in by_age_df["Not satisfied (%)"]],
              title="Share NOT satisfied with retirement financial planning, by age (raw)")
fig.update_traces(line=dict(color="#e55353", width=3), marker=dict(size=10),
                  textposition="top center", textfont=dict(size=11, color="#111827"))
fig.update_layout(yaxis_ticksuffix="%", yaxis_range=[0, 100])
FIGS["fig-satage"] = _clean(fig)

by_inc_series = df.groupby(df["HachnasaKoleletNeto"].map(INCOME_LABELS))[
    "HisachonPensyoni"].apply(lambda s: (s == 1).sum() / len(s) * 100).reindex(
    [INCOME_LABELS[k] for k in sorted(INCOME_LABELS)])
by_inc_df = by_inc_series.rename_axis("Income band").reset_index(name="Coverage (%)")
fig = px.bar(by_inc_df, x="Income band", y="Coverage (%)",
             title="Pension savings coverage by household net income (raw)",
             color="Coverage (%)", color_continuous_scale="Blues")
fig.update_layout(coloraxis_showscale=False, xaxis_tickangle=-30, yaxis_ticksuffix="%")
fig.update_traces(text=[f"{v:.0f}%" for v in by_inc_df["Coverage (%)"]],
                  textposition="outside", cliponaxis=False, textfont=dict(size=12))
FIGS["fig-coverage"] = _clean(fig, height=490)

opt_map_raw = {**OPTIMISM_LABELS, 888888: "Don't know (888888)"}
opt_order = [OPTIMISM_LABELS[k] for k in sorted(OPTIMISM_LABELS)] + ["Don't know (888888)"]
opt_counts = (df["OptimiyutKalkalit"].map(opt_map_raw).value_counts().reindex(opt_order)
              .rename_axis("Economic outlook").reset_index(name="Respondents"))
fig = px.pie(opt_counts, names="Economic outlook", values="Respondents",
             title="Economic optimism (raw)", hole=0.4,
             color_discrete_sequence=["#4CAF50", "#9E9E9E", "#E53935", "#6b7280"])
fig.update_traces(textinfo="percent+label+value", textfont_size=13)
FIGS["fig-optimism"] = _clean(fig)

# SCATTER: Age band vs Income band, size = count. Both variables are ordinal codes,
# so a scatter of joint distribution is shown as a bubble matrix.
grid = (pd.crosstab(df["Gil"], df["HachnasaKoleletNeto"]).stack()
        .rename("Count").reset_index())
grid = grid[grid["Count"] > 0].copy()
grid["Age group"] = grid["Gil"].map({**AGE_LABELS, 888888: "DK", 999999: "NA"})
grid["Income"] = grid["HachnasaKoleletNeto"].map({**INCOME_LABELS, 888888: "DK", 999999: "NA"})
scatter_ao = age_order + [x for x in ["DK", "NA"] if x in set(grid["Age group"])]
scatter_io = [INCOME_LABELS[k] for k in sorted(INCOME_LABELS)] + [
    x for x in ["DK", "NA"] if x in set(grid["Income"])]
fig = px.scatter(grid, x="Income", y="Age group", size="Count", color="Count",
                 color_continuous_scale="Viridis", size_max=42,
                 title="Age x Income scatter (bubble size = respondent count, raw)",
                 category_orders={"Income": scatter_io, "Age group": scatter_ao})
fig.update_layout(xaxis_tickangle=-30)
FIGS["fig-scatter"] = _clean(fig, height=520)

# --- Retirement financial planning satisfaction by age (heatmap-style)
# Retirement gauge is added separately (built at page-load in JS).

print("Plotly figures built:", len(FIGS))

In [ ]:
# 4b. Word cloud (financial context only)

# Each token appears in the cloud with a weight equal to its raw count in the survey.
# Tokens are drawn from finance-related variables only; no non-financial content is included.

def yes_count(var):
    return int((df[var] == 1).sum())

def nonmissing_count(var):
    return int(len(df) - (df[var].isin([888888, 999999])).sum())

FIN_WEIGHTS = {
    "Pension":               yes_count("HisachonPensyoni"),
    "Study Fund":            yes_count("HatavaKHishtalmut_wp"),
    "Keren Hishtalmut":      yes_count("HatavaKHishtalmut_wp"),
    "Retirement":            nonmissing_count("MerutzeTichnunFinPrisha"),
    "Bank":                  yes_count("HashkaotBank") + yes_count("HalvaaBank"),
    "Investments":           yes_count("HashkaotBank"),
    "Stocks":                yes_count("HashkaotBank"),
    "Bonds":                 yes_count("HashkaotBank"),
    "Loan":                  yes_count("HalvaaBank"),
    "Mortgage":              yes_count("HalvaaBank"),
    "Credit":                yes_count("InternetDohAshrai"),
    "Bitcoin":               yes_count("InternetShilemMatbeaDig"),
    "Crypto":                yes_count("InternetShilemMatbeaDig"),
    "Digital Currency":      yes_count("InternetShilemMatbeaDig"),
    "Income":                nonmissing_count("HachnasaKoleletNeto"),
    "Salary":                nonmissing_count("HachnasaAvoda"),
    "Savings":               yes_count("HishtameshHisachon"),
    "Management Fees":       nonmissing_count("HatavaKHishtalmut_wp"),
    "Compound Interest":     nonmissing_count("MerutzeTichnunFinPrisha"),
    "Inflation":             nonmissing_count("MerutzeKalkali"),
    "Budget":                nonmissing_count("NisharKesef"),
    "Household Expenses":    nonmissing_count("MichyaHachnasaMineches"),
    "Financial Planning":    nonmissing_count("MerutzeTichnunFinPrisha"),
    "Financial Decisions":   nonmissing_count("DerechHachlataFin"),
    "Economic Optimism":     nonmissing_count("OptimiyutKalkalit"),
    "Wealth":                nonmissing_count("MerutzeKalkali"),
    "Debt":                  yes_count("HalvaaBank"),
    "Tax":                   nonmissing_count("HachnasaAvoda"),
    "Portfolio":             yes_count("HashkaotBank"),
    "Diversification":       yes_count("HashkaotBank"),
    "Passive Income":        yes_count("HashkaotBank"),
    "Provident Fund":        yes_count("HatavaKHishtalmut_wp"),
}

wc = WordCloud(width=1200, height=520, background_color="white",
               colormap="viridis", prefer_horizontal=0.85,
               relative_scaling=0.55, min_font_size=14,
               max_words=len(FIN_WEIGHTS)).generate_from_frequencies(FIN_WEIGHTS)
buf = io.BytesIO()
wc.to_image().save(buf, format="PNG")
WORDCLOUD_B64 = base64.b64encode(buf.getvalue()).decode("ascii")
print("Word cloud generated,", len(WORDCLOUD_B64), "chars b64,", len(FIN_WEIGHTS), "financial terms")

In [ ]:
# 4c. Folium map of Israel: respondent counts per Nafa subdistrict (raw, no cleaning).

nafa_counts = df["nafa"].value_counts().to_dict()

# Fixed center + zoom sized so the whole surveyed area (Golan -> Beer Sheva)
# is visible in the 600px iframe. No fit_bounds - it over-zooms in some browsers.
m = folium.Map(location=[31.85, 35.10], zoom_start=8,
               tiles="cartodbpositron", control_scale=True, prefer_canvas=False,
               min_zoom=7, max_zoom=12,
               max_bounds=True)

# Bubbles: linear scaling but capped so the largest bubble (Tel Aviv, 1,093) doesn't
# swallow smaller neighbors like Ashkelon.
max_c = max(nafa_counts.values()) if nafa_counts else 1
counts_present = []

for code_val, meta in NAFA.items():
    n = int(nafa_counts.get(code_val, 0))
    if n == 0:
        continue
    # sqrt scaling: bubble AREA proportional to count (visually more accurate)
    radius = 6 + 22 * (n / max_c) ** 0.5
    popup_html = (
        f"<div style='font-size:13px;padding:2px 4px'>"
        f"<b style='font-size:14px;color:#1f4e9e'>{meta['name']}</b><br>"
        f"<span style='color:#6b7280'>Nafa code: {code_val}</span><br>"
        f"Respondents: <b>{n:,}</b><br>"
        f"Share of survey: <b>{n / len(df) * 100:.1f}%</b>"
        f"</div>"
    )
    folium.CircleMarker(
        location=[meta["lat"], meta["lon"]],
        radius=radius,
        color="#1f5fc4", weight=1.5,
        fill=True, fill_color="#2c7be5", fill_opacity=0.55,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=folium.Tooltip(
            f"<b>{meta['name']}</b>: {n:,} respondents "
            f"({n / len(df) * 100:.1f}%)",
            sticky=True,
        ),
    ).add_to(m)
    counts_present.append((meta["lat"], meta["lon"]))

# Do NOT call fit_bounds - Folium 0.20 tends to over-zoom to the tightest
# level that fits, which for a small country like Israel hits max_zoom.
# The zoom_start=8 above frames Golan through Beer Sheva at 600px height.

# Small legend anchored to the top-right corner of the map (Leaflet native).
legend_html = """
<div style="
  position:absolute; top:12px; right:12px; z-index:1000;
  background:#fff; padding:10px 14px; border-radius:8px;
  border:1px solid #e3e8ee; box-shadow:0 2px 6px rgba(0,0,0,.08);
  font-family:-apple-system,'Segoe UI',Arial,sans-serif; font-size:12px;
  color:#111827; max-width:220px;">
  <div style="font-weight:700;margin-bottom:4px;color:#1f4e9e">
    Respondents per subdistrict
  </div>
  <div style="color:#4b5563">Bubble size = respondent count (sqrt scale).
  Hover for the exact number, click for details.</div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

map_html_full = m.get_root().render()
MAP_SRCDOC = map_html_full
print("Folium map rendered:", len(MAP_SRCDOC), "chars for",
      len(counts_present), "subdistricts, max count:", max_c)

In [ ]:
# 4d. Overview HTML content (KPIs, reserved codes note, raw data preview, missing summary)

kpis = [("Respondents", f"{df.shape[0]:,}"), ("Variables", df.shape[1]),
        ("Survey year", "2024"), ("Memory", f"{df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")]
kpi_html = "".join(
    f"<div class='kpi'><div class='kpi-v'>{v}</div><div class='kpi-k'>{k}</div></div>"
    for k, v in kpis)

# Missing summary uses the reserved codes as-is (no replacement, no dropping): counts of
# rows that are 888888 or 999999 per variable.
raw_flag_missing = df.isin([888888, 999999])
missing_summary = pd.DataFrame({
    "reserved_code_rows": raw_flag_missing.sum(),
    "reserved_%": (raw_flag_missing.mean() * 100).round(1),
}).sort_values("reserved_%", ascending=False)

OVERVIEW_HTML = (
    f"<div class='kpis'>{kpi_html}</div>"
    "<p class='hint'><b>Reserved codes:</b> 888888 = don't know / refused, "
    "999999 = not applicable (question was skipped). These codes are kept as-is in every "
    "chart on this dashboard. No cleaning or imputation is done here (data cleaning is the "
    "responsibility of the data-preparation stage of the pipeline).</p>"
    f"<h3>Raw data preview (first 10 rows, first 40 of {df.shape[1]} columns)</h3>"
    "<div class='tablebox'>" + df.iloc[:10, :40].to_html() + "</div>"
    "<h3>Top 15 variables by reserved-code rate</h3>"
    "<div class='tablebox'>" + missing_summary.head(15).to_html() + "</div>"
)
print("Overview HTML ready,", len(OVERVIEW_HTML), "chars")

In [ ]:
# 5. Generate index.html (standalone dashboard page) and open it

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<meta name="apple-mobile-web-app-capable" content="yes">
<meta name="apple-mobile-web-app-status-bar-style" content="black-translucent">
<meta name="theme-color" content="#1f4e9e">
<title>Financial Literacy and Retirement Readiness</title>
<script>__PLOTLYJS__</script>
<style>
:root {
  --blue:#2c7be5; --blue-d:#1f5fc4; --green:#2fb380; --red:#e55353;
  --amber:#f6b73c; --bg:#f5f7fa; --card:#ffffff;
  --border:#e3e8ee; --text:#111827; --muted:#4b5563;
}
* { box-sizing:border-box; -webkit-tap-highlight-color:transparent; }
html, body { margin:0; padding:0; }
html { -webkit-text-size-adjust:100%; text-size-adjust:100%; }
body {
  font-family:-apple-system,BlinkMacSystemFont,'Segoe UI','Helvetica Neue',Arial,sans-serif;
  background:var(--bg); color:var(--text); line-height:1.5; font-size:15px;
  overflow-x:hidden;
}

/* Top bar */
.topbar {
  background:linear-gradient(135deg,#1f4e9e 0%,#2c7be5 100%);
  color:#fff; padding:24px 20px;
  padding-top:calc(24px + env(safe-area-inset-top));
  padding-left:calc(20px + env(safe-area-inset-left));
  padding-right:calc(20px + env(safe-area-inset-right));
  box-shadow:0 2px 8px rgba(31,78,158,.18);
  text-align:center;
}
.topbar h1 { margin:0 auto; font-size:24px; font-weight:700; letter-spacing:.2px; max-width:1180px; }
.topbar .sub { opacity:.9; font-size:14px; margin:4px auto 0; max-width:1180px; }

/* Layout */
.wrap { max-width:1180px; margin:0 auto; padding:24px 16px 60px; }
.card {
  background:var(--card); border:1px solid var(--border); border-radius:12px;
  padding:22px 22px; margin-bottom:18px; box-shadow:0 1px 3px rgba(16,24,40,.05);
}
.card h2 { margin:0 0 6px 0; font-size:20px; color:var(--text); font-weight:700; text-align:center; }
.card h3 { margin:22px 0 8px 0; font-size:16px; color:var(--text); font-weight:600; text-align:center; }
.card .lead { color:var(--muted); font-size:14px; margin:0 0 16px 0; text-align:center; }
.hint { background:#eef4ff; border-left:4px solid var(--blue);
        padding:12px 14px; border-radius:8px; margin:16px 0; color:#1f2937; font-size:14px; }

/* Quiz */
.quiz-intro { background:var(--blue); color:#fff; border:none; }
.quiz-intro h2 { color:#fff; }
.quiz-intro .lead { color:rgba(255,255,255,.9); }
.quiz-body { max-width:760px; margin:0 auto; }
.question { margin:20px 0 6px 0; }
.qtext { font-weight:600; margin-bottom:10px; color:var(--text); font-size:15px; }
.opt { display:block; padding:12px 14px; margin:6px 0; border:1.5px solid var(--border);
       border-radius:10px; cursor:pointer; background:#fbfcfe; transition:all .12s ease;
       user-select:none; min-height:44px; /* iOS tap target */ }
.opt:active { background:#e8f1ff; }
.opt:hover { border-color:var(--blue); background:#f0f6ff; }
.opt input { margin-right:11px; vertical-align:middle; transform:scale(1.15); }
.opt input:checked ~ span { font-weight:600; }
.opt:has(input:checked) { border-color:var(--blue); background:#e8f1ff; }
#progress { font-weight:600; color:var(--muted); margin:18px 0 12px 0;
            font-size:14px; text-align:center; }
.btn-row { text-align:center; }
.btn {
  background:var(--blue); color:#fff; border:none; border-radius:10px;
  padding:14px 36px; font-size:15px; font-weight:600; cursor:pointer;
  transition:background .12s ease; box-shadow:0 2px 6px rgba(44,123,229,.25);
  min-height:48px; /* iOS tap target */
}
.btn:hover { background:var(--blue-d); }
.btn:active { background:var(--blue-d); }
.errbox {
  display:none; margin-top:14px; padding:12px 16px;
  border-left:4px solid var(--red); background:#fdf1f1; border-radius:8px;
  color:#7f1d1d; font-size:14px; text-align:left;
}
.banner {
  padding:14px 18px; border-left:4px solid var(--green); background:#eefaf5;
  border-radius:10px; margin-bottom:20px; font-size:15px; color:#065f46;
  text-align:center;
  display:flex; flex-wrap:wrap; align-items:center; justify-content:center; gap:14px;
}
.banner .banner-text { flex:1 1 auto; min-width:200px; }

/* Back-to-quiz pill button */
.home-btn {
  background:#fff; border:1.5px solid var(--green); color:#065f46;
  border-radius:999px; padding:9px 18px; font-size:14px; font-weight:600;
  cursor:pointer; transition:all .12s ease; min-height:40px;
  display:inline-flex; align-items:center; gap:8px; flex-shrink:0;
}
.home-btn:hover { background:var(--green); color:#fff; }
.home-btn:active { background:#207f5b; color:#fff; }
.home-btn svg { width:16px; height:16px; }

/* Tabs - horizontally scrollable on mobile, centered on desktop */
.tabs {
  position:sticky; top:0; z-index:20; background:var(--bg);
  padding:12px 0 8px; display:flex; gap:8px;
  border-bottom:1px solid var(--border); margin-bottom:14px;
  overflow-x:auto; overflow-y:hidden;
  scrollbar-width:none; -ms-overflow-style:none;
  -webkit-overflow-scrolling:touch;
  justify-content:flex-start;
}
.tabs::-webkit-scrollbar { display:none; }
@media (min-width: 780px) { .tabs { flex-wrap:wrap; justify-content:center; overflow:visible; } }
.tabbtn {
  background:#fff; border:1.5px solid var(--border); border-radius:10px;
  padding:12px 18px; font-size:14px; cursor:pointer; color:#374151;
  transition:all .12s ease; font-weight:500; flex-shrink:0;
  min-height:44px; white-space:nowrap;
}
.tabbtn:hover { border-color:var(--blue); color:var(--blue); }
.tabbtn.active {
  background:var(--blue); border-color:var(--blue); color:#fff; font-weight:600;
  box-shadow:0 2px 6px rgba(44,123,229,.3);
}
.panel { display:none; }

/* KPIs - centered */
.kpis { display:flex; gap:14px; flex-wrap:wrap; margin-bottom:14px; justify-content:center; }
.kpi {
  flex:1 1 200px; max-width:260px; padding:16px 22px; background:#f0f6ff;
  border-left:4px solid var(--blue); border-radius:10px; text-align:center;
}
.kpi-v { font-size:24px; font-weight:700; color:var(--text); }
.kpi-k { color:var(--muted); font-size:13px; margin-top:2px; }

/* Tables */
.tablebox {
  overflow-x:auto; max-height:360px; overflow-y:auto;
  border:1px solid var(--border); border-radius:10px; margin-bottom:14px;
  -webkit-overflow-scrolling:touch;
}
table.dataframe { border-collapse:collapse; font-size:13px; width:100%; }
table.dataframe th, table.dataframe td {
  border:1px solid #e5e7eb; padding:6px 10px; text-align:left; white-space:nowrap;
}
table.dataframe th { background:#f3f6fa; position:sticky; top:0; font-weight:600; }
.ktable { border-collapse:collapse; font-size:14px; width:100%; }
.ktable th, .ktable td { border:1px solid #e5e7eb; padding:9px 12px; text-align:left; }
.ktable th { background:#f3f6fa; font-weight:600; }
.ok { color:var(--green); font-weight:700; }
.bad { color:var(--red); font-weight:700; }

/* Chart wrapper: CENTERED and FULL-WIDTH so no bars/labels get clipped */
.chart-wrap {
  margin:14px auto 24px;
  max-width:1100px;
  display:flex; flex-direction:column; align-items:stretch;
}
.chart-wrap .caption {
  color:var(--muted); font-size:13px; margin-top:4px; padding:0 4px; text-align:center;
}
.chart { width:100%; min-height:430px; }

/* Word cloud & map */
.wc-box { text-align:center; margin:6px auto 12px; max-width:1100px; }
.wc-box img { max-width:100%; height:auto; border-radius:10px; display:block; margin:0 auto; }
.map-box {
  border:1px solid var(--border); border-radius:10px; overflow:hidden;
  background:#fff; margin:0 auto; width:100%; max-width:1100px;
}
.map-box iframe { width:100%; min-width:100%; height:560px; border:0; display:block; }

/* Two-column layout - kept for legacy but now single-column everywhere so
   charts have enough room and never clip axis labels. */
.two-col { display:block; }

/* Model tab */
.dashed {
  border:2px dashed #9ca3af; border-radius:12px; background:#fafafa;
  padding:18px 22px; margin:12px auto; max-width:820px;
}
.dashed h3 { margin-top:0; color:var(--text); text-align:center; }

footer {
  color:var(--muted); font-size:13px; padding:14px 20px 32px;
  padding-bottom:calc(32px + env(safe-area-inset-bottom));
  text-align:center; background:var(--bg);
}

/* MOBILE tuning (iOS/Android portrait) */
@media (max-width: 640px) {
  body { font-size:14px; }
  .topbar { padding:20px 16px; padding-top:calc(20px + env(safe-area-inset-top)); }
  .topbar h1 { font-size:20px; }
  .topbar .sub { font-size:12.5px; }
  .wrap { padding:16px 12px 40px; }
  .card { padding:18px 14px; border-radius:10px; }
  .card h2 { font-size:18px; }
  .kpi { padding:14px 16px; }
  .kpi-v { font-size:20px; }
  .tabbtn { padding:11px 14px; font-size:13.5px; }
  .btn { width:100%; }
  .btn-row { text-align:stretch; }
  .chart { min-height:340px; }
  .map-box iframe { height:420px; }
  table.dataframe th, table.dataframe td { padding:5px 8px; font-size:12px; }
  .ktable th, .ktable td { padding:8px 8px; font-size:13px; }
}
</style>
</head>
<body>
<div class="topbar">
  <h1>Financial Literacy and Retirement Readiness</h1>
  <div class="sub">Social Survey 2024 - Israel - 6,907 respondents - Mid-project dashboard</div>
</div>
<div class="wrap">

  <!-- QUIZ ENTRY -->
  <div id="quiz-screen">
    <div class="card quiz-intro">
      <h2>Financial Literacy Quiz</h2>
      <p class="lead">Answer all 10 questions to enter the dashboard. Your answers also
      feed the Your Profile tab, which compares you against the 6,907 survey respondents.</p>
    </div>
    <div class="card">
      <div class="quiz-body">
        <div id="questions"></div>
        <hr style="border:none;border-top:1px solid var(--border);margin:20px 0 4px 0">
        <div id="progress"></div>
        <div class="btn-row"><button class="btn" onclick="submitQuiz()">Submit answers</button></div>
        <div id="quiz-error" class="errbox"></div>
      </div>
    </div>
  </div>

  <!-- DASHBOARD -->
  <div id="dashboard" style="display:none">
    <div id="score-banner" class="banner"></div>
    <div class="tabs">
      <button class="tabbtn" data-tab="overview" onclick="showTab('overview')">Overview</button>
      <button class="tabbtn" data-tab="demographics" onclick="showTab('demographics')">Demographics</button>
      <button class="tabbtn" data-tab="behavior" onclick="showTab('behavior')">Financial Behavior</button>
      <button class="tabbtn" data-tab="retirement" onclick="showTab('retirement')">Retirement Readiness</button>
      <button class="tabbtn" data-tab="profile" onclick="showTab('profile')">Your Profile</button>
      <button class="tabbtn" data-tab="model" onclick="showTab('model')">Model (Coming Soon)</button>
    </div>

    <div id="panel-overview" class="panel card">
      <h2>Overview</h2>
      <p class="lead">Structural view of the raw Social Survey 2024 data. Reserved codes
      (888888 and 999999) are kept as-is throughout the dashboard.</p>
      __OVERVIEW__
    </div>

    <div id="panel-demographics" class="panel card">
      <h2>Demographics</h2>
      <p class="lead">Who took the survey. Age, gender, income, education, and geographic
      distribution across Israel.</p>

      <div class="chart-wrap">
        <h3 style="margin-bottom:8px">Respondents across Israel (by CBS Nafa subdistrict)</h3>
        <div id="fig-map" class="map-box"></div>
        <div class="caption">Bubble AREA is proportional to respondent count (sqrt scale).
        <b>Hover</b> any bubble for the exact number, <b>click</b> for full details. The map
        is fixed on Israel; you can still zoom with the +/- controls.</div>
      </div>

      <div class="two-col">
        <div class="chart-wrap"><div id="fig-gender" class="chart"></div></div>
        <div class="chart-wrap"><div id="fig-age" class="chart"></div></div>
      </div>
      <div class="chart-wrap"><div id="fig-income" class="chart"></div></div>
      <div class="chart-wrap"><div id="fig-edu" class="chart"></div></div>

      <div class="chart-wrap">
        <div id="fig-heatmap" class="chart"></div>
        <div class="caption">Cross-tabulation of age band and net household income band.
        Cell values are raw respondent counts.</div>
      </div>
    </div>

    <div id="panel-behavior" class="panel card">
      <h2>Financial Behavior</h2>
      <p class="lead">How respondents behave with money, pensions, investments, and crypto.</p>

      <div class="chart-wrap"><div id="fig-behaviors" class="chart"></div></div>

      <div class="chart-wrap">
        <h3>Financial vocabulary present in the survey</h3>
        <div class="caption" style="margin:0 0 10px 0">Word cloud of finance-related
        concepts. Word size is proportional to the count of respondents whose data
        supports that concept.</div>
        <div class="wc-box">
          <img src="data:image/png;base64,__WORDCLOUD__" alt="Financial vocabulary word cloud">
        </div>
      </div>

      <div class="two-col">
        <div class="chart-wrap"><div id="fig-satisfaction" class="chart"></div></div>
        <div class="chart-wrap"><div id="fig-leftover" class="chart"></div></div>
      </div>
    </div>

    <div id="panel-retirement" class="panel card">
      <h2>Retirement Readiness</h2>
      <p class="lead">Cross-cuts that expose which slices of the population are least prepared.</p>

      <div class="chart-wrap"><div id="fig-satage" class="chart"></div></div>
      <div class="chart-wrap"><div id="fig-coverage" class="chart"></div></div>

      <div class="two-col">
        <div class="chart-wrap"><div id="fig-optimism" class="chart"></div></div>
        <div class="chart-wrap"><div id="fig-scatter" class="chart"></div></div>
      </div>
    </div>

    <div id="panel-profile" class="panel card">
      <h2>Your Profile</h2>
      <p class="lead">Built from your quiz answers. Compares you against the 6,907 raw
      survey respondents.</p>

      <div class="chart-wrap"><div id="fig-gauge" class="chart" style="min-height:360px"></div></div>
      <div class="chart-wrap"><div id="fig-profile" class="chart" style="min-height:480px"></div></div>

      <h3>Knowledge questions breakdown</h3>
      <div class="tablebox"><table class="ktable" id="ktable">
        <thead><tr><th>Question</th><th>Your answer</th><th>Correct answer</th><th>Result</th></tr></thead>
        <tbody></tbody>
      </table></div>
    </div>

    <div id="panel-model" class="panel card">
      <h2>Model (Coming Soon)</h2>
      <p class="lead">This section will show the retirement risk score and its explanation
      once the trained model artifacts land.</p>
      <div class="dashed"><h3>Personalized Retirement Risk Score</h3>
        <p><b>Coming soon.</b> Risk probability (0-100%) with a color-coded band, computed
        from quiz answers plus derived features.</p></div>
      <div class="dashed"><h3>SHAP Explanation</h3>
        <p><b>Coming soon.</b> Waterfall plot showing which answers pushed the score up
        and which pulled it down.</p></div>
      <div class="dashed"><h3>Action Recommendations</h3>
        <p><b>Coming soon.</b> Concrete next steps based on top negative contributors:
        open a pension account, switch Study Fund track, review management fees.</p></div>
    </div>
  </div>

</div>
<footer>
  Data source: Israeli Central Bureau of Statistics, Social Survey 2024 (Public Use File).
  Pipeline stage: raw data (no cleaning applied here).
</footer>

<script>
var FIGS = __FIGS__;
var QUIZ = __QUIZ__;
var MAP_SRCDOC = __MAP_SRCDOC__;
var userAnswers = null;
var rendered = {};

var TAB_FIGS = {
  demographics: ["fig-gender", "fig-age", "fig-income", "fig-edu", "fig-heatmap"],
  behavior: ["fig-behaviors", "fig-satisfaction", "fig-leftover"],
  retirement: ["fig-satage", "fig-coverage", "fig-optimism", "fig-scatter"],
  profile: ["fig-gauge", "fig-profile"]
};

function el(id) { return document.getElementById(id); }

function backToQuiz() {
  // Clear all radio selections
  document.querySelectorAll("#questions input[type=radio]").forEach(function (r) {
    r.checked = false;
  });
  // Reset state
  userAnswers = null;
  window.userScore = null;
  window.userKtot = null;
  rendered = {};
  // Clear error box and refresh progress counter
  var err = el("quiz-error"); if (err) { err.style.display = "none"; err.innerHTML = ""; }
  updateProgress();
  // Swap screens
  el("dashboard").style.display = "none";
  el("quiz-screen").style.display = "block";
  window.scrollTo({ top: 0, behavior: "smooth" });
}

function buildQuiz() {
  var box = el("questions");
  QUIZ.forEach(function (q, i) {
    var d = document.createElement("div"); d.className = "question";
    var t = document.createElement("div"); t.className = "qtext";
    t.textContent = (i + 1) + ". " + q.text; d.appendChild(t);
    q.options.forEach(function (opt, oi) {
      var lab = document.createElement("label"); lab.className = "opt";
      var inp = document.createElement("input");
      inp.type = "radio"; inp.name = q.id; inp.value = oi;
      inp.addEventListener("change", updateProgress);
      lab.appendChild(inp);
      var sp = document.createElement("span"); sp.textContent = opt; lab.appendChild(sp);
      d.appendChild(lab);
    });
    box.appendChild(d);
  });
  updateProgress();
}

function currentAnswers() {
  var a = {};
  QUIZ.forEach(function (q) {
    var sel = document.querySelector("input[name='" + q.id + "']:checked");
    a[q.id] = sel ? parseInt(sel.value, 10) : null;
  });
  return a;
}

function updateProgress() {
  var a = currentAnswers(), n = 0;
  QUIZ.forEach(function (q) { if (a[q.id] !== null) n++; });
  var p = el("progress");
  p.textContent = "Answered: " + n + " / " + QUIZ.length;
  p.style.color = (n === QUIZ.length) ? "#2fb380" : "#4b5563";
}

function submitQuiz() {
  var a = currentAnswers(), missing = [];
  QUIZ.forEach(function (q, i) { if (a[q.id] === null) missing.push(i + 1); });
  var err = el("quiz-error");
  if (missing.length) {
    err.style.display = "block";
    err.innerHTML = "<b>Cannot enter the dashboard yet.</b> Please answer question" +
      (missing.length > 1 ? "s" : "") + " " + missing.join(", ") + ".";
    return;
  }
  err.style.display = "none";
  userAnswers = a;
  var score = 0, ktot = 0;
  QUIZ.forEach(function (q) {
    if (q.kind === "knowledge") { ktot++; if (a[q.id] === q.correct) score++; }
  });
  window.userScore = score;
  window.userKtot = ktot;
  el("quiz-screen").style.display = "none";
  el("dashboard").style.display = "block";
  el("score-banner").innerHTML =
    "<div class='banner-text'><b>Dashboard unlocked.</b> Knowledge score: " +
    score + " / " + ktot + ". Open <b>Your Profile</b> for the population comparison.</div>" +
    "<button class='home-btn' onclick='backToQuiz()' aria-label='Back to quiz'>" +
    "<svg viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='2.2' " +
    "stroke-linecap='round' stroke-linejoin='round'>" +
    "<path d='M3 12L12 3l9 9'/><path d='M5 10v10h14V10'/>" +
    "</svg> Back to quiz</button>";
  buildKnowledgeTable(a);
  buildMap();
  rendered = {};  // Reset render cache so profile/gauge draw fresh with the current score
  showTab("overview");
  window.scrollTo(0, 0);
}

function showTab(name) {
  document.querySelectorAll(".tabbtn").forEach(function (b) {
    b.classList.toggle("active", b.dataset.tab === name);
  });
  document.querySelectorAll(".panel").forEach(function (p) {
    p.style.display = (p.id === "panel-" + name) ? "block" : "none";
  });
  (TAB_FIGS[name] || []).forEach(function (fid) {
    if (!rendered[fid]) {
      if (fid === "fig-profile") { buildProfileChart(); }
      else if (fid === "fig-gauge") { buildGauge(); }
      else {
        var f = FIGS[fid];
        Plotly.newPlot(fid, f.data, f.layout, { responsive: true, displaylogo: false });
      }
      rendered[fid] = true;
    } else {
      try { Plotly.Plots.resize(fid); } catch (e) {}
    }
  });
  // The Folium map lives inside an iframe; every time Demographics is shown
  // (including on tab switches after the container was resized), nudge Leaflet
  // to fit the new box width.
  if (name === "demographics") { setTimeout(refreshMap, 80); }
}

// Also refresh on window resize (rotate on mobile)
window.addEventListener("resize", function () { setTimeout(refreshMap, 80); });

function refreshMap() {
  // Force Leaflet to recalculate its container size (the iframe starts hidden
  // at the initial small size then expands with CSS - Leaflet needs a nudge).
  var iframe = document.querySelector('#fig-map iframe');
  if (!iframe || !iframe.contentWindow) return;
  var win = iframe.contentWindow;
  var mapKey = Object.keys(win).find(function (k) { return k.indexOf('map_') === 0; });
  var m = mapKey && win[mapKey];
  if (m && m.invalidateSize) { m.invalidateSize(); }
}

function buildMap() {
  var box = el("fig-map");
  if (box && !box.querySelector("iframe")) {
    var iframe = document.createElement("iframe");
    iframe.setAttribute("srcdoc", MAP_SRCDOC);
    iframe.setAttribute("loading", "eager");
    iframe.onload = function () {
      // Give Leaflet a beat, then force a resize so it fits the wide box.
      setTimeout(refreshMap, 60);
      setTimeout(refreshMap, 350);
    };
    box.appendChild(iframe);
  } else {
    refreshMap();
  }
}

function buildGauge() {
  var score = window.userScore, ktot = window.userKtot;
  if (!ktot) return;  // guard: only meaningful after quiz submission
  var pct = score / ktot * 100;
  var color = pct >= 80 ? "#2fb380" : pct >= 50 ? "#f6b73c" : "#e55353";
  var band = pct >= 80 ? "Excellent" : pct >= 50 ? "Good" : "Needs improvement";
  Plotly.newPlot("fig-gauge", [{
    type: "indicator", mode: "gauge+number",
    value: score,
    number: { suffix: " / " + ktot, font: { size: 44 } },
    title: { text: "<b>Financial Knowledge</b><br><span style='font-size:14px;color:" +
                    color + "'>" + band + "</span>",
             font: { size: 18 } },
    gauge: {
      axis: { range: [0, ktot], tickwidth: 1, tickcolor: "#374151" },
      bar: { color: color, thickness: 0.75 },
      bgcolor: "white", borderwidth: 1, bordercolor: "#e3e8ee",
      steps: [
        { range: [0, ktot * 0.5], color: "#fde8e8" },
        { range: [ktot * 0.5, ktot * 0.8], color: "#fff4dc" },
        { range: [ktot * 0.8, ktot], color: "#e2f5ec" }
      ]
    }
  }], { height: 340, margin: { t: 60, b: 20, l: 30, r: 30 },
        paper_bgcolor: "white", plot_bgcolor: "white" },
     { responsive: true, displaylogo: false });
}

function buildProfileChart() {
  if (!userAnswers) return;  // guard: only meaningful after quiz submission
  var beh = QUIZ.filter(function (q) { return q.kind === "behavior"; });
  var colors = beh.map(function (q) { return userAnswers[q.id] === 0 ? "#2c7be5" : "#e55353"; });
  Plotly.newPlot("fig-profile", [{
    type: "bar", orientation: "h",
    x: beh.map(function (q) { return q.popYes; }),
    y: beh.map(function (q) { return q.text; }),
    marker: { color: colors },
    text: beh.map(function (q) { return q.popYes.toFixed(1) + "%"; }),
    textposition: "outside",
    textfont: { size: 12 },
    cliponaxis: false
  }], {
    title: {
      text: "<b>You vs the population</b><br>" +
            "<span style='font-size:12px;color:#4b5563'>Bar = share of Yes among all 6,907 " +
            "respondents (raw). Blue = you answered Yes, red = you answered No.</span>",
      x: 0.5, xanchor: "center"
    },
    height: 480, margin: { t: 90, r: 60, b: 60 },  // let automargin size the left
    xaxis: { range: [0, 105], ticksuffix: "%",
             showgrid: false, zeroline: false, showline: true, linecolor: "#d1d5db",
             ticks: "outside", tickcolor: "#d1d5db" },
    yaxis: { autorange: "reversed", automargin: true,
             showgrid: false, zeroline: false, showline: false,
             ticks: "outside", tickcolor: "#d1d5db" },
    paper_bgcolor: "white", plot_bgcolor: "white"
  }, { responsive: true, displaylogo: false });
}

function buildKnowledgeTable(a) {
  var tbody = el("ktable").querySelector("tbody");
  tbody.innerHTML = "";
  QUIZ.forEach(function (q) {
    if (q.kind !== "knowledge") return;
    var ok = a[q.id] === q.correct;
    var tr = document.createElement("tr");
    [q.text, q.options[a[q.id]], q.options[q.correct]].forEach(function (v) {
      var td = document.createElement("td"); td.textContent = v; tr.appendChild(td);
    });
    var td = document.createElement("td");
    td.textContent = ok ? "Correct" : "Wrong";
    td.className = ok ? "ok" : "bad";
    tr.appendChild(td);
    tbody.appendChild(tr);
  });
}

buildQuiz();
</script>
</body>
</html>
"""

figs_payload = {name: json.loads(fig.to_json()) for name, fig in FIGS.items()}

def js_safe(s):
    # Neutralize any '</script>' inside a JSON string embedded in a <script> block.
    return s.replace("</script>", "<\\/script>").replace("</Script>", "<\\/Script>")

html = (HTML_TEMPLATE
        .replace("__PLOTLYJS__", get_plotlyjs())
        .replace("__OVERVIEW__", OVERVIEW_HTML)
        .replace("__FIGS__", js_safe(json.dumps(figs_payload)))
        .replace("__QUIZ__", js_safe(json.dumps(QUIZ)))
        .replace("__WORDCLOUD__", WORDCLOUD_B64)
        .replace("__MAP_SRCDOC__", js_safe(json.dumps(MAP_SRCDOC))))

OUT_HTML = os.path.abspath("index.html")
with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)
print(f"Wrote {OUT_HTML} ({os.path.getsize(OUT_HTML) / 1024**2:.1f} MB)")

if os.environ.get("DASHBOARD_NO_OPEN") != "1":
    webbrowser.open("file:///" + OUT_HTML.replace("\\", "/"))
    print("Opened in your default browser.")

---
**Data source:** Israeli Central Bureau of Statistics, Social Survey 2024 (Public Use File).
**Pipeline stage:** raw data. Cleaning, imputation and modeling happen in later pipeline
stages; the outputs will be plugged into the Model tab of `index.html`.